In [1]:
import sqlite3
import pandas as pd
import numpy as np

conn = sqlite3.connect('../data/recommender.db')

users_df = pd.read_sql_query("SELECT * FROM users", conn)
videos_df = pd.read_sql_query("SELECT * FROM videos", conn)
interactions_df = pd.read_sql_query("SELECT * FROM interactions", conn)

full_df = pd.read_sql_query('''
    SELECT i.*, u.age, u.language AS user_language, u.location,
           v.category, v.language AS video_language, v.duration_sec
    FROM interactions i
    JOIN users u ON i.user_id = u.user_id
    JOIN videos v ON i.video_id = v.video_id
''', conn)

print(full_df.shape)

(50, 13)


In [2]:
full_df['language_match'] = (full_df['user_language'] == full_df['video_language']).astype(int)
full_df['duration_norm'] = (full_df['duration_sec'] - full_df['duration_sec'].min()) / \
                             (full_df['duration_sec'].max() - full_df['duration_sec'].min())
full_df['hour'] = pd.to_datetime(full_df['timestamp']).dt.hour
full_df['is_night'] = ((full_df['hour'] >= 19) | (full_df['hour'] <= 5)).astype(int)

# User's historical preference for this category, EXCLUDING the current row (avoids leakage)
def category_affinity(row, df):
    others = df[(df['user_id'] == row['user_id']) & (df['video_id'] != row['video_id'])]
    same_cat = others[others['category'] == row['category']]
    if len(others) == 0:
        return 0.5
    return same_cat['liked'].sum() / len(others) if len(others) > 0 else 0.0

full_df['category_affinity'] = full_df.apply(lambda r: category_affinity(r, full_df), axis=1)

feature_cols = ['category_affinity', 'language_match', 'duration_norm', 'is_night']
X = full_df[feature_cols].values
y = full_df['liked'].values

print(pd.DataFrame(X, columns=feature_cols).head())

   category_affinity  language_match  duration_norm  is_night
0               0.25             1.0       0.000000       1.0
1               0.25             1.0       0.027027       1.0
2               0.00             1.0       1.000000       1.0
3               0.50             0.0       0.009009       1.0
4               0.00             1.0       0.459459       1.0


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut

loo = LeaveOneOut()
correct = 0
for train_idx, test_idx in loo.split(X):
    model = LogisticRegression()
    model.fit(X[train_idx], y[train_idx])
    pred = model.predict(X[test_idx])
    correct += (pred[0] == y[test_idx][0])

loo_accuracy = correct / len(X)
print(f"Leave-One-Out CV Accuracy: {loo_accuracy:.2%}")

# Final model trained on ALL data, used for actual recommendations
final_model = LogisticRegression()
final_model.fit(X, y)

for name, coef in zip(feature_cols, final_model.coef_[0]):
    print(f"{name}: {coef:.3f}")

Leave-One-Out CV Accuracy: 80.00%
category_affinity: 0.216
language_match: 1.797
duration_norm: 0.356
is_night: -0.233


In [4]:
def predict_recommendations(user_id, users_df, videos_df, full_df, model, current_hour, n=3):
    user_row = users_df[users_df['user_id'] == user_id].iloc[0]
    watched = full_df[full_df['user_id'] == user_id]['video_id'].tolist()
    candidates = videos_df[~videos_df['video_id'].isin(watched)].copy()

    if len(watched) == 0:
        # cold start — fall back to popularity in user's language
        popularity = full_df.groupby('video_id')['liked'].sum()
        candidates['popularity'] = candidates['video_id'].map(popularity).fillna(0)
        candidates['lang_match'] = candidates['language'] == user_row['language']
        return candidates.sort_values(['lang_match', 'popularity'], ascending=[False, False]).head(n)

    rows = []
    for _, v in candidates.iterrows():
        lang_match = int(v['language'] == user_row['language'])
        dur_norm = (v['duration_sec'] - videos_df['duration_sec'].min()) / \
                   (videos_df['duration_sec'].max() - videos_df['duration_sec'].min())
        hist = full_df[(full_df['user_id'] == user_id) & (full_df['category'] == v['category'])]
        affinity = hist['liked'].sum() / len(hist) if len(hist) > 0 else 0.5
        is_night = int(current_hour >= 19 or current_hour <= 5)
        rows.append([affinity, lang_match, dur_norm, is_night])

    X_candidates = np.array(rows)
    scores = model.predict_proba(X_candidates)[:, 1]  # probability of "liked"
    candidates['predicted_score'] = scores

    return candidates.sort_values('predicted_score', ascending=False).head(n)[['video_id', 'category', 'language', 'predicted_score']]

for uid in range(1, 6):
    print(f"\nUser {uid} recommendations (8 PM):")
    print(predict_recommendations(uid, users_df, videos_df, full_df, final_model, current_hour=20, n=3))


User 1 recommendations (8 PM):
   video_id category language  predicted_score
9        10    Music  English         0.766068
4         5  Cooking  English         0.766068
2         3   Sports  English         0.759105

User 2 recommendations (8 PM):
    video_id category language  predicted_score
10        11   Gaming  English         0.389469
8          9    Music  Spanish         0.381492
9         10    Music  English         0.376965

User 3 recommendations (8 PM):
    video_id category language  predicted_score
10        11   Gaming  English         0.415470
6          7     Tech  English         0.395193
11        12   Gaming    Hindi         0.392327

User 4 recommendations (8 PM):
   video_id category language  predicted_score
6         7     Tech  English         0.779568
0         1   Comedy  English         0.769920
2         3   Sports  English         0.759105

User 5 recommendations (8 PM):
    video_id category language  predicted_score
11        12   Gaming    Hindi  

In [5]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import numpy as np

y_true = []
y_pred = []

for train_idx, test_idx in loo.split(X):
    model = LogisticRegression()
    model.fit(X[train_idx], y[train_idx])
    pred = model.predict(X[test_idx])
    y_true.append(y[test_idx][0])
    y_pred.append(pred[0])

y_true = np.array(y_true)
y_pred = np.array(y_pred)

cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print("                Predicted: Not Liked   Predicted: Liked")
print(f"Actual: Not Liked      {cm[0][0]:>10}          {cm[0][1]:>10}")
print(f"Actual: Liked          {cm[1][0]:>10}          {cm[1][1]:>10}")

print(f"\nAccuracy:  {accuracy_score(y_true, y_pred):.2%}")
print(f"Precision: {precision_score(y_true, y_pred, zero_division=0):.2%}")
print(f"Recall:    {recall_score(y_true, y_pred, zero_division=0):.2%}")
print(f"F1 Score:  {f1_score(y_true, y_pred, zero_division=0):.2%}")

Confusion Matrix:
                Predicted: Not Liked   Predicted: Liked
Actual: Not Liked              12                   6
Actual: Liked                   4                  28

Accuracy:  80.00%
Precision: 82.35%
Recall:    87.50%
F1 Score:  84.85%


In [6]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import cross_validate

# Surprise expects: user_id, item_id, rating — we'll use 'liked' as the rating (0 or 1)
surprise_df = full_df[['user_id', 'video_id', 'liked']].copy()

reader = Reader(rating_scale=(0, 1))
data = Dataset.load_from_df(surprise_df, reader)

svd_model = SVD()
results = cross_validate(svd_model, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)

print(f"\nSVD Average RMSE: {results['test_rmse'].mean():.3f}")
print(f"SVD Average MAE: {results['test_mae'].mean():.3f}")

Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.4421  0.3715  0.4322  0.3879  0.3855  0.4038  0.0279  
MAE (testset)     0.4049  0.3518  0.4247  0.3582  0.3008  0.3681  0.0435  
Fit time          0.00    0.00    0.00    0.00    0.00    0.00    0.00    
Test time         0.00    0.00    0.00    0.00    0.00    0.00    0.00    

SVD Average RMSE: 0.404
SVD Average MAE: 0.368


In [7]:
from surprise.model_selection import train_test_split as surprise_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

trainset, testset = surprise_split(data, test_size=0.2, random_state=42)
svd_model.fit(trainset)
predictions = svd_model.test(testset)

y_true_svd = [1 if p.r_ui >= 0.5 else 0 for p in predictions]
y_pred_svd = [1 if p.est >= 0.5 else 0 for p in predictions]

print("SVD (Surprise) metrics:")
print(f"Accuracy:  {accuracy_score(y_true_svd, y_pred_svd):.2%}")
print(f"Precision: {precision_score(y_true_svd, y_pred_svd, zero_division=0):.2%}")
print(f"Recall:    {recall_score(y_true_svd, y_pred_svd, zero_division=0):.2%}")
print(f"F1 Score:  {f1_score(y_true_svd, y_pred_svd, zero_division=0):.2%}")

SVD (Surprise) metrics:
Accuracy:  70.00%
Precision: 62.50%
Recall:    100.00%
F1 Score:  76.92%


In [8]:
comparison = pd.DataFrame({
    'Model': ['Popularity Baseline', 'Your Logistic Regression', 'Surprise SVD'],
    'Accuracy': ['—', f'{accuracy_score(y_true, y_pred):.0%}', f'{accuracy_score(y_true_svd, y_pred_svd):.0%}'],
    'Precision': ['0%', f'{precision_score(y_true, y_pred, zero_division=0):.0%}', f'{precision_score(y_true_svd, y_pred_svd, zero_division=0):.0%}'],
    'Recall': ['—', f'{recall_score(y_true, y_pred, zero_division=0):.0%}', f'{recall_score(y_true_svd, y_pred_svd, zero_division=0):.0%}'],
})
print(comparison)

                      Model Accuracy Precision Recall
0       Popularity Baseline        —        0%      —
1  Your Logistic Regression      80%       82%    88%
2              Surprise SVD      70%       62%   100%
